In [15]:
from pathlib import Path
import os, sys, subprocess, sqlite3, json
import pandas as pd

# >>> Your canonical Douyin project folder (based on your scan)
REPO_DIR = Path("/Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media")

if not REPO_DIR.exists():
    raise FileNotFoundError(f"REPO_DIR does not exist: {REPO_DIR}")

if not (REPO_DIR / "main.py").exists():
    raise FileNotFoundError(f"main.py not found in: {REPO_DIR}  (you may be pointing to the wrong folder)")

os.chdir(REPO_DIR)

print("✅ REPO_DIR:", REPO_DIR)
print("✅ CWD:", Path.cwd())
print("✅ Python:", sys.executable)


✅ REPO_DIR: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media
✅ CWD: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media
✅ Python: /opt/anaconda3/bin/python


In [17]:
print("🗑️ Clearing previous data...")

for fname in ["data.db", "data.xlsx"]:
    fp = REPO_DIR / fname
    if fp.exists():
        fp.unlink()
        print("   ✅ Deleted:", fp)
    else:
        print("   ⚠️ Not found:", fp)

print("✅ Reset done.")


🗑️ Clearing previous data...
   ✅ Deleted: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media/data.db
   ⚠️ Not found: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media/data.xlsx
✅ Reset done.


In [19]:
douyin_urls = """
https://www.iesdouyin.com/share/video/6862533859125251340/?region=CN&mid=6862534011185892109&u_code=0&did=MS4wLjABAAAA_oswJqZmU3zCGUyu8OVAC1UU2mCfmc4viCaQE_FNvdMR9u0Icg6C4KNfO5gmMhFF&iid=MS4wLjABAAAA0wvEGyff1ADjiPCblRQIWtvkG7eNV8LYyZ5_BPkYmSd4qCnN--z9LfPLty-TjygW&with_sec_did=1&titleType=title&share_sign=n02R2vZ5QCppS8kyhh0M5mtEAoEqelWcGWdKq65lS64-&share_version=110900&ts=1716365321&from_aid=2955&from_ssr=1
https://www.iesdouyin.com/share/video/7124980639866064136/?region=CN&mid=7124980721214688008&u_code=0&did=MS4wLjABAAAAFs6wBLkYsgEPdS0toRIdQ4FNZVrRttrZ-gldZ-56LSK3eaHcPqyi3HGlBt4b5WY5&iid=MS4wLjABAAAAcvmrmllesJPQC9ZqkE5KrO3Pltag8SfeuErDYO_x-KbAD5NpQEh0Owx11QG1GDhO&with_sec_did=1&titleType=title&share_sign=F7KrmxLyM3IblwEJJ_M6lglp5BE8bB9DnZKXa92CDkU-&share_version=110900&ts=1716365931&from_aid=2955&from_ssr=1
"""

urls = [u.strip() for u in douyin_urls.strip().splitlines() if u.strip()]
urls_txt = REPO_DIR / "urls.txt"

if not urls:
    raise ValueError("No URLs found. Paste URLs into douyin_urls.")

urls_txt.write_text("\n".join(urls), encoding="utf-8")
print(f"✅ Saved {len(urls)} URLs to {urls_txt}")


✅ Saved 2 URLs to /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media/urls.txt


In [21]:
MAX_COMMENTS = 500
SCROLL_COUNT = 30
HEADLESS = False  # Set True after first successful run if desired

douyin_file = REPO_DIR / "scrape_douyin_post.py"
backup_file = REPO_DIR / "scrape_douyin_post.py.backup"

# Backup original
if douyin_file.exists():
    backup_file.write_text(douyin_file.read_text(encoding="utf-8"), encoding="utf-8")
    print("✅ Backup saved:", backup_file)

douyin_content = f'''import asyncio
from datetime import datetime
from playwright.async_api import async_playwright
import json
import os
import utils
import config
from logging_config import get_logger

logger = get_logger()
SESSION_FILE = "douyin_session.json"

async def extract_details_new(page):
    details = {{
        "title": None,
        "content": None,
        "like_count": None,
        "comment_count": None,
        "share_count": None,
        "publish_time": None
    }}

    try:
        title = await page.locator('xpath=(//div[@data-e2e="user-info"]/div[2]/a/div)[2]').inner_text()
        details["title"] = title.split("\\n")[0]
    except Exception as e:
        logger.warning(f"Title error: {{e}}")

    try:
        details["content"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[1]/div/h1').inner_text()
    except Exception as e:
        logger.warning(f"Content error: {{e}}")

    try:
        details["like_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div[1]/div[1]/span').inner_text()
    except Exception as e:
        logger.warning(f"Like count error: {{e}}")

    try:
        details["comment_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div/div[2]/span').inner_text()
    except Exception as e:
        logger.warning(f"Comment count error: {{e}}")

    try:
        details["share_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div/div[4]/span').inner_text()
    except Exception as e:
        logger.warning(f"Share count error: {{e}}")

    try:
        publish_time = await page.locator('span[data-e2e="detail-video-publish-time"]').inner_text()
        publish_time = publish_time.replace('发布时间：', '').strip()
        dt_object = datetime.strptime(publish_time, '%Y-%m-%d %H:%M')
        details["publish_time"] = dt_object.strftime('%Y-%m-%d %H:%M:%S')
    except Exception as e:
        logger.warning(f"Publish time error: {{e}}")

    return details


async def extract_comments(page, max_comments={MAX_COMMENTS}):
    comments = []
    previous_count = 0
    no_change_count = 0

    for i in range({SCROLL_COUNT}):
        try:
            await page.evaluate("window.scrollBy(0, 800)")
            await page.wait_for_timeout(2000)

            comment_elems = await page.locator('[data-e2e="comment-item"]').all()
            current_count = len(comment_elems) if comment_elems else 0

            if current_count == previous_count:
                no_change_count += 1
                if no_change_count >= 5:
                    break
            else:
                no_change_count = 0

            previous_count = current_count

            if current_count >= max_comments:
                break
        except Exception as e:
            logger.warning(f"Scroll error: {{e}}")

    selectors = ['[data-e2e="comment-item"]', '.comment-item', '[class*="comment"]']
    comment_elements = None
    for sel in selectors:
        try:
            comment_elements = await page.locator(sel).all()
            if comment_elements:
                break
        except:
            continue

    if not comment_elements:
        return comments

    for elem in comment_elements[:max_comments]:
        try:
            comments.append(await elem.inner_text())
        except:
            continue

    return comments


async def scrape_post(url, conn):
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless={HEADLESS},
            args=['--start-maximized']
        )

        if os.path.exists(SESSION_FILE):
            context = await browser.new_context(storage_state=SESSION_FILE, viewport={{"width": 1920, "height": 1080}}, ignore_https_errors=True)
        else:
            context = await browser.new_context(viewport={{"width": 1920, "height": 1080}}, ignore_https_errors=True)

        page = await context.new_page()

        try:
            await page.goto(url)
            await page.wait_for_timeout(5000)

            # Attempt dismiss login popup
            try:
                close_btn = page.locator('xpath=//div[contains(text(), "登录后免费畅享高清视频")]/following-sibling::div[1]')
                if await close_btn.count() > 0:
                    await close_btn.click()
                    await page.wait_for_timeout(2000)
            except:
                pass

            # Save session
            if not os.path.exists(SESSION_FILE):
                try:
                    await context.storage_state(path=SESSION_FILE)
                except:
                    pass

            details = await extract_details_new(page)
            raw_comments = await extract_comments(page, max_comments={MAX_COMMENTS})
            comments = utils.extract_douyin_comments_from_text(raw_comments)
            comments_json = json.dumps(comments, ensure_ascii=False) if comments else None

            item = [{{
                'unnamed': None,
                'user_name': details['title'].strip() if details['title'] else None,
                'publication_date': details['publish_time'].strip() if details['publish_time'] else None,
                'content': details['content'].strip() if details['content'] else None,
                'shared_count': utils.chinese_unit_to_number(details['share_count'].strip()) if details['share_count'] else 0,
                'comment_count': utils.chinese_unit_to_number(details['comment_count'].strip()) if details['comment_count'] else 0,
                'like_count': utils.chinese_unit_to_number(details['like_count'].strip()) if details['like_count'] else 0,
                'link1': url,
                'link2': None,
                'content_segmented': None,
                'is_agriculture_related': None,
                'index_number': None,
                'comments': comments_json
            }}]

            utils.insert_data(conn, config.table_name, item)

        except Exception as e:
            logger.error(f"Error scraping {{url}}: {{e}}")
        finally:
            await browser.close()
'''

douyin_file.write_text(douyin_content, encoding="utf-8")
print("✅ Updated:", douyin_file)
print(f"   MAX_COMMENTS={MAX_COMMENTS}, SCROLL_COUNT={SCROLL_COUNT}, HEADLESS={HEADLESS}")


✅ Backup saved: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media/scrape_douyin_post.py.backup
✅ Updated: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media/scrape_douyin_post.py
   MAX_COMMENTS=500, SCROLL_COUNT=30, HEADLESS=False


In [23]:
urls_txt = REPO_DIR / "urls.txt"
if not urls_txt.exists():
    raise FileNotFoundError(f"urls.txt not found: {urls_txt}")

urls = [u.strip() for u in urls_txt.read_text(encoding="utf-8").splitlines() if u.strip()]
douyin_urls = [u for u in urls if "douyin" in u.lower()]

if not douyin_urls:
    raise ValueError("No Douyin URLs in urls.txt")

print(f"📋 Found {len(douyin_urls)} Douyin URLs")

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["PYTHONUTF8"] = "1"

result = subprocess.run([sys.executable, "main.py"], cwd=str(REPO_DIR), env=env)
print("✅ main.py finished. Return code:", result.returncode)

db_path = REPO_DIR / "data.db"
print("📁 data.db exists in REPO_DIR?", db_path.exists(), db_path)


📋 Found 2 Douyin URLs
INFO - connect to database successful: data.db
INFO - Prepare to scrape url: https://www.iesdouyin.com/share/video/6862533859125251340/?region=CN&mid=6862534011185892109&u_code=0&did=MS4wLjABAAAA_oswJqZmU3zCGUyu8OVAC1UU2mCfmc4viCaQE_FNvdMR9u0Icg6C4KNfO5gmMhFF&iid=MS4wLjABAAAA0wvEGyff1ADjiPCblRQIWtvkG7eNV8LYyZ5_BPkYmSd4qCnN--z9LfPLty-TjygW&with_sec_did=1&titleType=title&share_sign=n02R2vZ5QCppS8kyhh0M5mtEAoEqelWcGWdKq65lS64-&share_version=110900&ts=1716365321&from_aid=2955&from_ssr=1
INFO - Prepare to scrape url: https://www.iesdouyin.com/share/video/7124980639866064136/?region=CN&mid=7124980721214688008&u_code=0&did=MS4wLjABAAAAFs6wBLkYsgEPdS0toRIdQ4FNZVrRttrZ-gldZ-56LSK3eaHcPqyi3HGlBt4b5WY5&iid=MS4wLjABAAAAcvmrmllesJPQC9ZqkE5KrO3Pltag8SfeuErDYO_x-KbAD5NpQEh0Owx11QG1GDhO&with_sec_did=1&titleType=title&share_sign=F7KrmxLyM3IblwEJJ_M6lglp5BE8bB9DnZKXa92CDkU-&share_version=110900&ts=1716365931&from_aid=2955&from_ssr=1
✅ main.py finished. Return code: 0
📁 data.db exis

In [25]:
db_path = REPO_DIR / "data.db"
if not db_path.exists():
    raise FileNotFoundError(f"data.db not found at {db_path}. Step 4 didn’t create it.")

conn = sqlite3.connect(db_path)

tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
print("Tables:", tables)

# Prefer 'posts' if exists; otherwise use first table
table = "posts" if "posts" in tables else (tables[0] if tables else None)
if table is None:
    conn.close()
    raise RuntimeError("No tables found in data.db (empty DB).")

df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
conn.close()

print("✅ Using table:", table)
print("📈 Total rows:", len(df))
display(df.head())


Tables: ['posts', 'sqlite_sequence']
✅ Using table: posts
📈 Total rows: 2


,id,unnamed,user_name,publication_date,content,shared_count,comment_count,like_count,link1,link2,content_segmented,is_agriculture_related,index_number,comments
0,1,None,桥城都匀,2020-08-19 13:36:00,一起来看看非洲刚果的工人们是如何分工的#农民工 #非洲 #效率,3531.0,5385.0,45000.0,https://www.iesdouyin.com/share/video/68625338...,None,None,None,None,"[{""username"": ""╰☆微笑の孤叶☆╮"", ""content"": ""分工明确"", ..."
1,2,None,宣威融媒,2022-07-27 19:23:00,云南又来上分了！农村喜事大家围着桌子一起打跳，透过屏幕满满的代入感。#结婚 #现场实拍 #云...,1180.0,3009.0,43300.0,https://www.iesdouyin.com/share/video/71249806...,None,None,None,None,"[{""username"": ""楚兰"", ""content"": """", ""time"": ""20..."


In [27]:
db_path = REPO_DIR / "data.db"
if not db_path.exists():
    raise FileNotFoundError("data.db not found. Run Step 4 first.")

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["PYTHONUTF8"] = "1"

result = subprocess.run([sys.executable, "export_excel_data.py"], cwd=str(REPO_DIR), env=env)
print("Return code:", result.returncode)

xlsx_path = REPO_DIR / "data.xlsx"
if not xlsx_path.exists():
    raise FileNotFoundError("Export did not create data.xlsx. Check export_excel_data.py output/logs.")

print("✅ Exported:", xlsx_path)
print("📁 Size (KB):", xlsx_path.stat().st_size / 1024)

df_xlsx = pd.read_excel(xlsx_path)
display(df_xlsx.head())


Processing table: posts
Processing table: sqlite_sequence
[OK] Done! Export excel path: data.xlsx
Return code: 0
✅ Exported: /Users/paullam/Downloads/chinese-social-media-scrape-updated/duoyin/scrape_chinese_social_media/data.xlsx
📁 Size (KB): 7.681640625


,Unnamed: 0,User.name,Publication.date,Content,Share,Comment,Like,Link1,Link2,content_segmented,is_agriculture_related,No.,Comments
0,NaN,桥城都匀,2020-08-19 13:36:00,一起来看看非洲刚果的工人们是如何分工的#农民工 #非洲 #效率,3531,5385,45000,https://www.iesdouyin.com/share/video/68625338...,NaN,NaN,NaN,NaN,"[{""username"": ""╰☆微笑の孤叶☆╮"", ""content"": ""分工明确"", ..."
1,NaN,宣威融媒,2022-07-27 19:23:00,云南又来上分了！农村喜事大家围着桌子一起打跳，透过屏幕满满的代入感。#结婚 #现场实拍 #云...,1180,3009,43300,https://www.iesdouyin.com/share/video/71249806...,NaN,NaN,NaN,NaN,"[{""username"": ""楚兰"", ""content"": """", ""time"": ""20..."
